## 4.0 Introduction

In Lecture 3 we explored the Iris dataset with full knowledge of the species labels — we used them to color scatter plots and compute per-species centroids. But consider what we actually *used* the labels for: nothing about the centroids themselves required knowing which flower was which. We just looked at the data and computed means.

This lecture asks a natural question: **what if we didn't have the labels?** Could we still find the three clusters?

The answer is yes (mostly), and the algorithm that does it is called **k-means**. It discovers cluster structure from the data matrix $X$ alone — no label vector $\mathbf{y}$ required. This puts it in a category of methods called **unsupervised learning**: there is no ground truth to train against, only the geometry of the data.

After we cluster, we face a second problem: the Iris data lives in $\mathbb{R}^4$, and we cannot directly plot it. We will use **Principal Component Analysis (PCA)** to project it down to $\mathbb{R}^2$ so we can actually see the cluster structure we found.

By the end of this lecture you will be able to:

- Implement Lloyd's algorithm step by step on a small dataset
- Define and compute $J^{\text{clust}}$, the k-means objective function
- Run k-means on standardized Iris data using `sklearn`
- Use PCA to visualize high-dimensional clusters by projecting them into lower dimensional spaces
- Evaluate k-means results against known species labels.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 4.1 Supervised vs. Unsupervised Learning

Everything we have done so far in this class has been **supervised** in a loose sense: we had labels. In Lecture 3 we used the species column to color our scatter plots and compute per-species centroids. The labels told us what was "correct."

**Supervised learning** is the general setting where a model trains on labeled data — it can check its own work by comparing predictions to known outcomes.

**Unsupervised learning** is the setting where there are no labels. The algorithm must find structure in the data on its own. There is no external standard to check against.

k-means is an unsupervised method. It groups data vectors into $k$ clusters by finding centroids that minimize total distance — without ever seeing the species column. We use Iris precisely because we *do* have labels, which lets us check after the fact whether k-means found something real. In a true unsupervised problem, no such check exists.

**Question.** Think back to the prediction model from Lectures 1 and 2: $\hat{y} = w_1 x + w_2 z$. Is that supervised or unsupervised? What plays the role of the "labels" there?

## 4.2 The k-Means Objective Function

To group data vectors, we need a mathematical definition of what "good grouping" means. k-means uses **distance to the centroid** as its measure of quality.

Let $\mathbf{x}_i \in \mathbb{R}^p$ be the $i^{th}$ row vector (one observation) in the dataset, for $i = 1, \ldots, n$. Let $\mathbf{c}_1, \ldots, \mathbf{c}_k$ be $k$ **centroid vectors** — one per group. Each data vector $\mathbf{x}_i$ is assigned to exactly one group; call its centroid $\mathbf{c}_{g_i}$, where $g_i \in \{1, \ldots, k\}$ is the group index for observation $i$.

The **k-means objective function** is the mean squared distance from each data vector to its assigned centroid:

$$J^{\text{clust}} = \frac{1}{n} \sum_{i=1}^{n} \|\mathbf{x}_i - \mathbf{c}_{g_i}\|^2$$

**Question.** What is a centroid vector $\mathbf{c}_{g_i}$?

**Question.** What does $\|\mathbf{x}_i - \mathbf{c}_{g_i}\|$ measure geometrically? If every data vector sat exactly on its centroid, what would $J^{\text{clust}}$ be?

**Question.** The formula uses $\|\cdot\|^2$ rather than $\|\cdot\|$. Why might squaring the distance be mathematically convenient? (Hint: think about what squaring does to the norm formula from Lecture 2.)

## 4.3 Lloyd's Algorithm

We cannot search all possible groupings of $n$ points into $k$ clusters — the number of possibilities is astronomically large even for modest $n$ and $k$. Instead, k-means uses a simple iterative procedure known as **Lloyd's algorithm** that is guaranteed to decrease $J^{\text{clust}}$ at every step.

**Lloyd's Algorithm:**

1. Choose $k$. Randomly initialize $k$ centroid vectors $\mathbf{c}_1, \ldots, \mathbf{c}_k$.
2. **Assign:** For each data vector $\mathbf{x}_i$, compute $\|\mathbf{x}_i - \mathbf{c}_j\|^2$ for all $j = 1, \ldots, k$. Assign $\mathbf{x}_i$ to the group with the nearest centroid.
3. **Update:** Recompute each centroid $\mathbf{c}_j$ as the mean of all data vectors currently assigned to group $j$.
4. Compute $J^{\text{clust}}$.
5. Repeat steps 2–4 until $J^{\text{clust}}$ stops changing — the algorithm has **converged**.

**Definition.** An algorithm has **converged** when repeating its steps produces no further change in the objective function.

**Question.** Why is step 3 (recomputing $\mathbf{c}_j$ as a mean of the group) guaranteed to decrease $J^{\text{clust}}$ or leave it unchanged? Think about what the centroid is minimizing for its assigned group — you computed this exact object in Lecture 3.

## 4.4 k-Means From Scratch

Before using `sklearn`, we'll run Lloyd's algorithm by hand on a small dataset — the same approach we took with standardization in Lecture 3: derive it manually first, then confirm with the library.

Here is the dataset:

| $A$ | $B$ |
|-----|-----|
| 1   | 1   |
| 1   | 0   |
| 0   | 2   |
| 2   | 4   |
| 3   | 5   |

We have $n = 5$ data vectors in $\mathbb{R}^2$, and we will use $k = 2$ groups. Assume both variables are already on the same scale (so no standardization is needed here).

Initial centroids: $\mathbf{c}_0 = \mathbf{x}_0 = [1, 1]$ and $\mathbf{c}_1 = \mathbf{x}_2 = [0, 2]$.

In [ ]:
# Data matrix -- each row is one observation vector x_i
X_small = np.array([[1, 1],
                    [1, 0],
                    [0, 2],
                    [2, 4],
                    [3, 5]])

n = X_small.shape[0]  # number of data vectors

# Extract individual row vectors for readability
x_0 = X_small[0, :]
x_1 = X_small[1, :]
x_2 = X_small[2, :]
x_3 = X_small[3, :]
x_4 = X_small[4, :]

In [ ]:
# A reusable helper for plotting points and centroids at any iteration.
# group_labels is a list of length n assigning each x_i to group 0 or 1.
def plot_iteration(X_small, group_labels, c_0, c_1, title):
    colors = ['steelblue', 'tomato']
    fig, ax = plt.subplots(figsize=(5, 5))
    for i, xi in enumerate(X_small):
        g = group_labels[i]
        ax.scatter(xi[0], xi[1], color=colors[g],
                   edgecolors='white', linewidth=0.5, s=80, zorder=3)
        ax.annotate(f'$\\mathbf{{x}}_{i}$', xy=(xi[0], xi[1]),
                    xytext=(5, 5), textcoords='offset points', fontsize=10)
    ax.scatter(c_0[0], c_0[1], color=colors[0], marker='*', s=350,
               edgecolors='black', linewidth=0.8, zorder=4,
               label=f'$c_0$ = {np.round(c_0, 2)}')
    ax.scatter(c_1[0], c_1[1], color=colors[1], marker='*', s=350,
               edgecolors='black', linewidth=0.8, zorder=4,
               label=f'$c_1$ = {np.round(c_1, 2)}')
    ax.set_xlabel('$A$')
    ax.set_ylabel('$B$')
    ax.set_title(title)
    ax.legend(fontsize=9)
    ax.grid(True, linestyle='--', alpha=0.4)
    plt.tight_layout()
    plt.show()

def plot_raw(X_small, title):
    """Plot the raw data points with no assignments or centroids."""
    fig, ax = plt.subplots(figsize=(5, 5))
    for i, xi in enumerate(X_small):
        ax.scatter(xi[0], xi[1], color='gray',
                   edgecolors='white', linewidth=0.5, s=80, zorder=3)
        ax.annotate(f'$\\mathbf{{x}}_{i}$', xy=(xi[0], xi[1]),
                    xytext=(5, 5), textcoords='offset points', fontsize=10)
    ax.set_xlabel('$A$')
    ax.set_ylabel('$B$')
    ax.set_title(title)
    ax.grid(True, linestyle='--', alpha=0.4)
    plt.tight_layout()
    plt.show()

In [ ]:
# Initialize centroids
c_0 = x_0.copy()  # centroid for group 0
c_1 = x_2.copy()  # centroid for group 1

print(f'Initial c_0: {c_0}')
print(f'Initial c_1: {c_1}')

In [ ]:
# Plot the raw data before any assignments — no colors, no centroids
plot_raw(X_small, 'Raw data — no assignments yet')

### Iteration 1

**Step 2 — Assign:** Compute the squared distance from each $\mathbf{x}_i$ to each centroid. Assign each point to its nearest centroid.

**Step 3 — Update:** Recompute each centroid as the mean of its assigned points.

**Step 4:** Compute $J_1$.

In [ ]:
# Step 2: Assign — compute squared distances and assign each x_i to its nearest centroid
# We use np.linalg.norm(v)**2 to compute ||v||^2 -- the squared Euclidean distance

print('Distances: [to c_0, to c_1] -> assigned group')
group_labels_1 = []
for i, xi in enumerate([x_0, x_1, x_2, x_3, x_4]):
    d0 = np.linalg.norm(xi - c_0)**2
    d1 = np.linalg.norm(xi - c_1)**2
    g = 0 if d0 <= d1 else 1
    group_labels_1.append(g)
    print(f'  x_{i}: [{d0:.4f}, {d1:.4f}] -> group {g}')

In [ ]:
# Step 3: Update — recompute centroids as the mean of each group's assigned vectors
# group 0: {x_0, x_1, x_2}   group 1: {x_3, x_4}

c_0 = np.mean(np.array([x_0, x_1]), axis=0)  # column-wise mean
c_1 = np.mean(np.array([x_2, x_3, x_4]), axis=0)

print(f'Updated c_0: {c_0}')
print(f'Updated c_1: {c_1}')

**Question.** `np.mean(..., axis=0)` computes the column-wise mean of a matrix — you saw this in Lecture 3 when we computed species centroids. Confirm by hand that `c_0` is the mean of $\mathbf{x}_0$, $\mathbf{x}_1$, and $\mathbf{x}_2$.

In [ ]:
# Step 4: Compute J_1
J_1 = (np.linalg.norm(x_0 - c_0)**2 +
       np.linalg.norm(x_1 - c_0)**2 +
       np.linalg.norm(x_2 - c_1)**2 +
       np.linalg.norm(x_3 - c_1)**2 +
       np.linalg.norm(x_4 - c_1)**2) / n

print(f'J_1: {J_1:.4f}')

In [ ]:
plot_iteration(X_small, group_labels_1, c_0, c_1,
               f'Iteration 1 ($J_1$ = {J_1:.4f})')

**Question.** The centroids moved from their initial positions (the data points $\mathbf{x}_0$ and $\mathbf{x}_2$) to the means of their assigned groups. Do the updated centroids look like natural "centers" of their respective clusters?

### Iteration 2

In [ ]:
# Step 2: Re-assign with updated centroids
print('Distances: [to c_0, to c_1] -> assigned group')
group_labels_2 = []
for i, xi in enumerate([x_0, x_1, x_2, x_3, x_4]):
    d0 = np.linalg.norm(xi - c_0)**2
    d1 = np.linalg.norm(xi - c_1)**2
    g = 0 if d0 <= d1 else 1
    group_labels_2.append(g)
    print(f'  x_{i}: [{d0:.4f}, {d1:.4f}] -> group {g}')

In [ ]:
# Step 3: Update centroids (if assignments haven't changed, centroids won't either)
c_0 = np.mean(np.array([x_0, x_1, x_2]), axis=0)
c_1 = np.mean(np.array([x_3, x_4]), axis=0)

# Step 4: Compute J_2
J_2 = (np.linalg.norm(x_0 - c_0)**2 +
       np.linalg.norm(x_1 - c_0)**2 +
       np.linalg.norm(x_2 - c_0)**2 +
       np.linalg.norm(x_3 - c_1)**2 +
       np.linalg.norm(x_4 - c_1)**2) / n

print(f'J_1: {J_1:.4f}')
print(f'J_2: {J_2:.4f}')
print(f'Assignments changed? {group_labels_1 != group_labels_2}')

In [ ]:
plot_iteration(X_small, group_labels_2, c_0, c_1,
               f'Iteration 2 — ($J_2$ = {J_2:.4f})')

### Iteration 3: Check for Convergence

In [ ]:
# Step 2: Re-assign with updated centroids
print('Distances: [to c_0, to c_1] -> assigned group')
group_labels_3 = []
for i, xi in enumerate([x_0, x_1, x_2, x_3, x_4]):
    d0 = np.linalg.norm(xi - c_0)**2
    d1 = np.linalg.norm(xi - c_1)**2
    g = 0 if d0 <= d1 else 1
    group_labels_3.append(g)
    print(f'  x_{i}: [{d0:.4f}, {d1:.4f}] -> group {g}')

In [ ]:
# Step 3: Update centroids (if assignments haven't changed, centroids won't either)
c_0 = np.mean(np.array([x_0, x_1, x_2]), axis=0)
c_1 = np.mean(np.array([x_3, x_4]), axis=0)

# Step 4: Compute J_2
J_3 = (np.linalg.norm(x_0 - c_0)**2 +
       np.linalg.norm(x_1 - c_0)**2 +
       np.linalg.norm(x_2 - c_0)**2 +
       np.linalg.norm(x_3 - c_1)**2 +
       np.linalg.norm(x_4 - c_1)**2) / n

print(f'J_1: {J_1:.4f}')
print(f'J_2: {J_2:.4f}')
print(f'J_3: {J_3:.4f}')
print(f'Assignments changed? {group_labels_2 != group_labels_3}')

**The algorithm has converged.** Assignments did not change between iteration 1 and iteration 2. When assignments don't change, centroids don't change, and $J^{\text{clust}}$ doesn't change — the algorithm has nothing left to do.

**Final result:** Group 0 = $\{\mathbf{x}_0, \mathbf{x}_1, \mathbf{x}_2\}$, Group 1 = $\{\mathbf{x}_3, \mathbf{x}_4\}$.

**Question.** What is the smallest $J^{\text{clust}}$ could ever be? What value of $k$ would achieve it, and why would that be a useless model?

## 4.5 k-Means on Iris with `sklearn`

Running Lloyd's algorithm by hand was instructive, but impractical for real datasets. `sklearn`'s `KMeans` runs the full algorithm — including multiple random restarts — in a single call.

First, let's reload the Iris data and standardize it exactly as we did in Lecture 3.

In [ ]:
from sklearn.preprocessing import StandardScaler

url = 'https://raw.githubusercontent.com/ajorgen1/Linear-Algebra-With-Applications/main/datasets/iris.csv'
df = pd.read_csv(url)

X = df[['sepal_length', 'sepal_width', 'petal_length', 'petal_width']].to_numpy()
y = pd.Categorical(df['species']).codes   # 0=setosa, 1=versicolor, 2=virginica
species_names = df['species'].unique()
feature_names = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']

scaler = StandardScaler()
X_std = scaler.fit_transform(X)

print('X_std shape:', X_std.shape)
print('Column norms (should all be sqrt(150)):',
      np.round([np.linalg.norm(X_std[:, j]) for j in range(4)], 4))

### Ensemble Models and `n_init`

Because Lloyd's algorithm starts with randomly initialized centroids, different starting points can lead to different final groupings. This is a real problem — the algorithm is only guaranteed to find a *local* minimum of $J^{\text{clust}}$, not the global one.

The standard solution is an **ensemble model**: run the algorithm $m$ independent times with different random starts, then keep the solution with the lowest $J^{\text{clust}}$. The `n_init` parameter controls how many independent runs are used.

We also set `random_state` to make results reproducible — it seeds the random number generator so the same initialization sequence is used every time.

**Question.** Why do we run multiple models and pick the best, rather than just running one model with more iterations?

In [ ]:
# Run k-means: 3 clusters, 10 random restarts, reproducible seed
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=3, n_init=10, random_state=0)
labels = kmeans.fit_predict(X_std)

print('Cluster labels (first 150):', labels[:150])
print('Cluster label counts:', np.bincount(labels))

In [ ]:
# The learned centroids (in standardized space)
centers = kmeans.cluster_centers_
print('Cluster centroids (standardized):')
print(np.round(centers, 4))

**Question.** Setosa is typically recovered perfectly. Versicolor and Virginica are harder to separate — they overlap in feature space. Does this match what you observed in the Lecture 3 scatter plots?

## 4.6 The Visualization Problem

We have cluster labels. Now we want to plot the results. But $\mathbf{x}_i \in \mathbb{R}^4$ — we cannot directly scatter-plot 4-dimensional data. In Lecture 3 we worked around this by picking two features to plot (petal length vs. petal width). That's fine when you know which two features to look at, but what if you don't? And what if there's structure that doesn't show up cleanly in any two-feature slice?

What we really want is a method that finds **the two directions that capture the most variation in the data** — directions that together give the best 2D summary of the full 4D dataset.

## 4.7 What Is PCA?

We have 150 Iris flowers, each described by four measurements. That means each flower is a point in 4-dimensional space — impossible to draw. But most of the interesting structure in the data (the three species clusters) probably doesn't require all four dimensions to see. PCA finds a way to compress the data down to 2 dimensions while losing as little information as possible.

The key idea is **spread**. Imagine shining a flashlight on a cloud of points from different angles and watching the shadow it casts on a wall. Some angles produce a big, spread-out shadow that shows the shape of the cloud well. Other angles collapse everything into a tight blob and lose the structure entirely. PCA finds the angle — the direction — that produces the most spread-out shadow. That is the **first principal component**. It then finds the best perpendicular direction for a second axis, and so on.

The result is a new 2D coordinate system — one axis pointing in the direction of maximum spread, the other in the direction of maximum remaining spread — and every flower gets a new $(x, y)$ coordinate in that system. That 2D picture is the best possible flat summary of the 4D data.

A few things worth noting:

- The new axes (principal components) are not the original features. They are *combinations* of all four measurements, chosen to maximize spread.
- We standardize before running PCA for the same reason we standardize before k-means: if one feature has a much larger scale, it will dominate the spread calculation and the first principal component will mostly just point in that feature's direction.
- PCA does not use the species labels. Like k-means, it is unsupervised — it only looks at the geometry of the data.

**Question.** In Lecture 3 we made scatter plots by picking two features to put on the axes. How is that different from what PCA does? Which approach do you think would give a better picture of the cluster structure, and why?

## 4.8 PCA with `sklearn`

Finding the principal components requires solving a variance-maximization problem that involves the **covariance matrix** of $X_\text{std}$ — a $4 \times 4$ matrix that encodes how the four features vary together. Computing its eigenvectors gives the principal component directions. We will not derive this in full here (it belongs with the deeper linear algebra of Lecture 9), but `sklearn`'s `PCA` does it for us exactly.

What `PCA(n_components=2).fit_transform(X_std)` returns is the $150 \times 2$ matrix of projection coordinates. Each row $\mathbf{v}_i \in \mathbb{R}^2$ is flower $i$ projected into the plane of maximum variance — a 2D representation we can actually plot.

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
V = pca.fit_transform(X_std)   # V has shape (150, 2)

print('V shape:', V.shape)
print(f'First 3 rows of V:')
print(np.round(V[:3, :], 4))

In [ ]:
# How much variance does each component explain?
evr = pca.explained_variance_ratio_
print(f'PC1 explains {evr[0]*100:.1f}% of total variance')
print(f'PC2 explains {evr[1]*100:.1f}% of total variance')
print(f'Together:    {sum(evr)*100:.1f}%')

**Question.** Two dimensions explain a large fraction of the total variance. What does this tell you about the intrinsic dimensionality of the Iris dataset — even though the data matrix $X$ has 4 columns?

## 4.9 Visualizing k-Means Results via PCA

Now we have everything we need: k-means cluster labels from Section 4.5, and PCA coordinates from Section 4.8. Plotting $Y$ colored by cluster label gives us a 2D picture of the 4D cluster structure.

We'll make three plots side by side:

1. PCA coordinates colored by **k-means cluster label** — what the algorithm found without labels
2. PCA coordinates colored by **true species label** — the ground truth
3. The **petal length vs. petal width** plot from Lecture 3 — for comparison

In [ ]:
colors = ['steelblue', 'tomato', 'seagreen']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Plot 1: PCA colored by k-means labels
ax = axes[0]
for c in range(3):
    mask = (labels == c)
    ax.scatter(V[mask, 0], V[mask, 1], color=colors[c],
               label=f'Cluster {c}', alpha=0.7, edgecolors='white', linewidth=0.5)
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_title('k-Means clusters (PCA view)')
ax.legend(fontsize=9)

# Plot 2: PCA colored by true species
ax = axes[1]
for k, name in enumerate(species_names):
    mask = (y == k)
    ax.scatter(V[mask, 0], V[mask, 1], color=colors[k],
               label=name, alpha=0.7, edgecolors='white', linewidth=0.5)
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_title('True species (PCA view)')
ax.legend(fontsize=9)

# Plot 3: petal length vs petal width (Lecture 3 style)
ax = axes[2]
for k, name in enumerate(species_names):
    mask = (y == k)
    ax.scatter(X_std[mask, 2], X_std[mask, 3], color=colors[k],
               label=name, alpha=0.7, edgecolors='white', linewidth=0.5)
ax.set_xlabel('petal_length (standardized)')
ax.set_ylabel('petal_width (stdandardized)')
ax.set_title('Two-feature slice (Lecture 3 view)')
ax.legend(fontsize=9)

plt.suptitle('Iris: k-Means Clusters and PCA Visualization', fontsize=13)
plt.tight_layout()
plt.show()

**Question.** Compare the left two plots. The cluster assignments (left) and the true species (center) should look very similar. Where do they differ? Cross-reference with the cross-tabulation from Section 4.5.

**Question.** Compare the center and right plots. The PCA view (center) uses two directions optimized to show spread in the full 4D data. The two-feature slice (right) uses only petal length and petal width. Which gives a cleaner separation between species? Why might this be?

## 4.10 Why Standardize Before k-Means?

k-means assigns each point to its nearest centroid using Euclidean distance — $\|\mathbf{x}_i - \mathbf{c}_j\|$. This means that features with **larger numerical ranges will dominate the distance calculation**, regardless of whether they are actually more informative. 

**Question.** While larger numerical ranges have a greater affect on the distance calculation, what will be more influential in assignment to groups: the "smaller" scaled variable or the larger one?

Recall from Lecture 3: before standardization, the four Iris features had very different norms. A 1 cm difference in sepal length and a 1 cm difference in petal width contribute identically to the raw distance — but those features have very different natural scales of variation.

In [ ]:
# Run k-means on the RAW (unstandardized) data and compare
kmeans_raw = KMeans(n_clusters=3, n_init=10, random_state=0)
labels_raw = kmeans_raw.fit_predict(X)

# Cross-tabulation for raw clustering
print('Raw data clustering:')
for k, name in enumerate(species_names):
    row = [np.sum((y == k) & (labels_raw == j)) for j in range(3)]
    print(f'{name:>15}: {row}')

print()
print('Standardized data clustering:')
for k, name in enumerate(species_names):
    row = [np.sum((y == k) & (labels == j)) for j in range(3)]
    print(f'{name:>15}: {row}')

**Question.** Do the two clusterings agree? If the raw clustering does worse, which features do you think dominated the distance calculation — the ones with larger or smaller standard deviations?

**Looking ahead.** This lecture introduced two fundamental unsupervised techniques. In Homework 4 you will apply both to the Palmer Penguins dataset — standardize, cluster, reduce, visualize, and evaluate. In later lectures we'll return to PCA more formally when we have the tools (eigenvalues and SVD) to derive why the principal component directions are what they are.